## Feature generation

In [1]:
import pandas as pd
import numpy as np
import re
from rdkit.Chem.Descriptors import ExactMolWt
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, EState, GraphDescriptors, MolSurf
import skfp.fingerprints as sfp

C:\Users\adamk\Desktop\politechnika\sem5\problem_classes\visco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
visco = pd.read_csv('data/viscoelastic_cleared.csv', keep_default_na=False, na_values='')

In [27]:
visco.info()

<class 'pandas.DataFrame'>
RangeIndex: 8152 entries, 0 to 8151
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8152 non-null   str    
 1   type_surfactant1    8152 non-null   str    
 2   smiles_surfactant1  8152 non-null   str    
 3   c_surfactant1       8152 non-null   float64
 4   id_surfactant2      8152 non-null   str    
 5   type_surfactant2    8152 non-null   str    
 6   smiles_surfactant2  8152 non-null   str    
 7   c_surfactant2       2726 non-null   float64
 8   id_additive1        8152 non-null   str    
 9   smiles_additive1    8152 non-null   str    
 10  c_additive1         5522 non-null   float64
 11  id_additive2        8152 non-null   str    
 12  smiles_additive2    8152 non-null   str    
 13  c_additive2         1333 non-null   float64
 14  temperature         8152 non-null   int64  
 15  viscoelasticity     8152 non-null   float64
 16  g_0              

In [28]:
visco.head()

,id_surfactant1,type_surfactant1,smiles_surfactant1,c_surfactant1,id_surfactant2,type_surfactant2,smiles_surfactant2,c_surfactant2,id_additive1,smiles_additive1,...,id_additive2,smiles_additive2,c_additive2,temperature,viscoelasticity,g_0,g_min,omega_cr,tau_r,eta_0
0,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,0.0,NaN,NaN,NaN,NaN,0.02
1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,4.0,2.70,150.0,0.01,0.03
2,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,4.5,1.20,10.0,0.10,0.45
3,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,5.0,0.50,0.8,1.25,6.25
4,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,None,None,NaN,25,1.0,4.2,0.35,1.5,0.67,2.80


In [29]:
# Tail length
def extract_tail_c_number(smiles: str):
    # For now just count n of leading C's
    if smiles == 'None':
        return np.nan
    n_c = re.match(r"^(C+)", smiles)
    return len(n_c.group(1)) if n_c else 0


# Surfactant type is already in the data but further more advance type can be added later(look in README)

def extract_molecular_weight(smiles: str):
    if smiles == 'None':
        return np.nan
    mol = Chem.MolFromSmiles(smiles)
    return ExactMolWt(mol)


def generate_chen(smiles: str):
    if smiles == 'None':
        return pd.Series({
            "chen_MolLogP": None,
            "chen_Chi1v": None,
            "chen_VSA_EState7": None,
            "chen_VSA_EState8": None,
            "chen_qed": None,
            "chen_FpDensityMorgan1": None,
            "chen_SMR_VSA5": None,
            "chen_PEOE_VSA6": None,
            "chen_MinPartialCharge": None,
            "chen_BertzCT": None,
            "chen_VSA_EState1": None,
            "chen_MaxAbsEStateIndex": None,
            "chen_Balaban": None,
            "chen_SlogP_VSA2": None,
            "chen_HallKierAlpha": None,
            "chen_EState_VSA5": None,
            "chen_MaxPartialCharge": None,
            "chen_PEOE_VSA7": None,
            "chen_PEOE_VSA8": None
        })

    mol = Chem.MolFromSmiles(smiles)

    return pd.Series({
        "chen_MolLogP": Descriptors.MolLogP(mol),
        "chen_Chi1v": GraphDescriptors.Chi1v(mol),
        "chen_VSA_EState7": EState.EState_VSA.EState_VSA7(mol),
        "chen_VSA_EState8": EState.EState_VSA.EState_VSA8(mol),
        "chen_qed": QED.qed(mol),
        "chen_FpDensityMorgan1": Descriptors.FpDensityMorgan1(mol),
        "chen_SMR_VSA5": MolSurf.SMR_VSA5(mol),
        "chen_PEOE_VSA6": MolSurf.PEOE_VSA6(mol),
        "chen_MinPartialCharge": Descriptors.MinPartialCharge(mol),
        "chen_BertzCT": GraphDescriptors.BertzCT(mol),
        "chen_VSA_EState1": EState.EState_VSA.EState_VSA1(mol),
        "chen_MaxAbsEStateIndex": EState.MaxAbsEStateIndex(mol),
        "chen_Balaban": GraphDescriptors.BalabanJ(mol),
        "chen_SlogP_VSA2": MolSurf.SlogP_VSA2(mol),
        "chen_HallKierAlpha": Descriptors.HallKierAlpha(mol),
        "chen_EState_VSA5": EState.EState_VSA.EState_VSA5(mol),
        "chen_MaxPartialCharge": Descriptors.MaxPartialCharge(mol),
        "chen_PEOE_VSA7": MolSurf.PEOE_VSA7(mol),
        "chen_PEOE_VSA8": MolSurf.PEOE_VSA8(mol)
    })

# Requires .transform() - due  to representation it's not really suitable to put this into pandas(numpy better?)
# features = sfp.PhysiochemicalPropertiesFingerprint(fp_size=256)

In [30]:
visco['surfactant1_tail_length'] = visco['smiles_surfactant1'].apply(extract_tail_c_number)
visco['surfactant2_tail_length'] = visco['smiles_surfactant2'].apply(extract_tail_c_number)
visco['surfactant1_mol_weight'] = visco['smiles_surfactant1'].apply(extract_molecular_weight)
visco['surfactant2_mol_weight'] = visco['smiles_surfactant2'].apply(extract_molecular_weight)

In [31]:
visco.head()

,id_surfactant1,type_surfactant1,smiles_surfactant1,c_surfactant1,id_surfactant2,type_surfactant2,smiles_surfactant2,c_surfactant2,id_additive1,smiles_additive1,...,viscoelasticity,g_0,g_min,omega_cr,tau_r,eta_0,surfactant1_tail_length,surfactant2_tail_length,surfactant1_mol_weight,surfactant2_mol_weight
0,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,0.0,NaN,NaN,NaN,NaN,0.02,12,12.0,332.163339,313.249118
1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,1.0,4.0,2.70,150.0,0.01,0.03,12,12.0,332.163339,313.249118
2,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,1.0,4.5,1.20,10.0,0.10,0.45,12,12.0,332.163339,313.249118
3,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,1.0,5.0,0.50,0.8,1.25,6.25,12,12.0,332.163339,313.249118
4,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,1.0,4.2,0.35,1.5,0.67,2.80,12,12.0,332.163339,313.249118


In [32]:
chen_columns = [
    "chen_MolLogP",
    "chen_Chi1v",
    "chen_VSA_EState7",
    "chen_VSA_EState8",
    "chen_qed",
    "chen_FpDensityMorgan1",
    "chen_SMR_VSA5",
    "chen_PEOE_VSA6",
    "chen_MinPartialCharge",
    "chen_BertzCT",
    "chen_VSA_EState1",
    "chen_MaxAbsEStateIndex",
    "chen_Balaban",
    "chen_SlogP_VSA2",
    "chen_HallKierAlpha",
    "chen_EState_VSA5",
    "chen_MaxPartialCharge",
    "chen_PEOE_VSA7",
    "chen_PEOE_VSA8"
]

In [33]:
surf1_chen_columns = [ 'surfactant1_' + column for column in chen_columns]
surf2_chen_columns = [ 'surfactant2_' + column for column in chen_columns]

In [34]:
visco[surf1_chen_columns] = visco['smiles_surfactant1'].apply(generate_chen)
visco[surf2_chen_columns] = visco['smiles_surfactant2'].apply(generate_chen)

In [35]:
visco.head()

,id_surfactant1,type_surfactant1,smiles_surfactant1,c_surfactant1,id_surfactant2,type_surfactant2,smiles_surfactant2,c_surfactant2,id_additive1,smiles_additive1,...,surfactant2_chen_BertzCT,surfactant2_chen_VSA_EState1,surfactant2_chen_MaxAbsEStateIndex,surfactant2_chen_Balaban,surfactant2_chen_SlogP_VSA2,surfactant2_chen_HallKierAlpha,surfactant2_chen_EState_VSA5,surfactant2_chen_MaxPartialCharge,surfactant2_chen_PEOE_VSA7,surfactant2_chen_PEOE_VSA8
0,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,295.628638,6.09324,11.575318,3.274503,32.137605,-1.1,51.993423,0.307938,6.420822,19.3864
1,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,295.628638,6.09324,11.575318,3.274503,32.137605,-1.1,51.993423,0.307938,6.420822,19.3864
2,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,295.628638,6.09324,11.575318,3.274503,32.137605,-1.1,51.993423,0.307938,6.420822,19.3864
3,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,295.628638,6.09324,11.575318,3.274503,32.137605,-1.1,51.993423,0.307938,6.420822,19.3864
4,SLES,anionic,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+],395.0,LED1A,zwitterionic,CCCCCCCCCCCC(=O)NCCC[N+](C)C(=O)[O-],55.0,NaCl,[Cl-].[Na+],...,295.628638,6.09324,11.575318,3.274503,32.137605,-1.1,51.993423,0.307938,6.420822,19.3864


In [36]:
visco.to_csv('data/viscoelastic_expert_chen.csv', index=False)